<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Load-the-modules" data-toc-modified-id="Load-the-modules-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Load the modules</a></span><ul class="toc-item"><li><span><a href="#Define-the-function" data-toc-modified-id="Define-the-function-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Define the function</a></span></li></ul></li><li><span><a href="#Time-matrix-multiplication-without-jit" data-toc-modified-id="Time-matrix-multiplication-without-jit-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Time matrix multiplication without <code>jit</code></a></span></li><li><span><a href="#Try-jit" data-toc-modified-id="Try-jit-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Try <code>jit</code></a></span></li><li><span><a href="#Correct-answer?" data-toc-modified-id="Correct-answer?-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Correct answer?</a></span></li></ul></div>

# CS319 Week 12: `jit`

An example of using just-in-time compilation.
For 2526-CS319 Week 12. This will not work on `cloudjupyter`, but will work on Colab: https://colab.research.google.com/

## Load the modules

In [1]:
import numpy as np
import time
from numba import jit

### Define the function

In [2]:
def MatMat(A,B):
    N = len(A)
    C = np.zeros((N,N), dtype=A.dtype)
    for i in range(N):
        for j in range(N):
            for k in range(N):
                C[i][j]+=A[i,k]*B[k,j]
    return C 

## Time matrix multiplication without `jit`

In [3]:
N = 400;
A = np.random.rand(N,N)

In [4]:
start = time.time()
C1 = MatMat(A,A)
MatMat_time = time.time() - start
print(f"MatMat for {N}-by-{N} matrix took {MatMat_time :.3f} seconds")

MatMat for 400-by-400 matrix took 32.793 seconds


Sometimes (see following slides) code can be faster when run a second time. Let's check if that is the case here (I don't think it will be)

In [5]:
start = time.time()
C1 = MatMat(A,A)
MatMat_time = time.time() - start
print(f"MatMat for {N}-by-{N} matrix took {MatMat_time :.3f} seconds")

MatMat for 400-by-400 matrix took 33.092 seconds


## Try `jit`
Now let's decorate so that the `just-in-time` compiler is called.

In [6]:
@jit(parallel=True, fastmath=True)
def MatMat_jit(A,B):
    N = len(A)
    C = np.zeros((N,N), dtype=A.dtype)
    for i in range(N):
        for j in range(N):
            for k in range(N):
                C[i][j]+=A[i,k]*B[k,j]
    return C 

Run the decorated function first time (which will cause compilation)

In [7]:
start = time.time()
C2 = MatMat_jit(A,A)
MatMat_jit_time = time.time() - start
print(f"MatMat_jit for {N}-by-{N} matrix took {MatMat_jit_time :.3f} seconds")

MatMat_jit for 400-by-400 matrix took 0.805 seconds


Now run it a second time, so that we use the compiled version.

In [8]:
start = time.time()
C2 = MatMat_jit(A,A)
MatMat_jit_time = time.time() - start
print(f"MatMat_jit for {N}-by-{N} matrix took {MatMat_jit_time :.3f} seconds")

MatMat_jit for 400-by-400 matrix took 0.080 seconds


Of course, this particular example is best done in `numpy`:

In [9]:
start = time.time()
C3 = A@A # use NumPy
Numpy_time = time.time() - start
print(f"A@A  for {N}-by-{N} matrix took {Numpy_time :.3f} seconds")

A@A  for 400-by-400 matrix took 0.021 seconds


## Correct answer?
Check we are getting the right answer:

In [11]:
print(f"Difference for use slow and jit:  {np.linalg.norm(C1-C2):10.4e}")
print(f"Difference for use numpy and jit: {np.linalg.norm(C3-C2):10.4e}")

Difference for use slow and jit:  9.4505e-13
Difference for use numpy and jit: 8.0790e-13
